In [ ]:
# 1. 라이브러리 불러오기
import tensorflow as tf
import numpy as np
import re

# 2. 데이터 파일 다운로드
path_to_file = tf.keras.utils.get_file(
    'toji.txt',
    'https://raw.githubusercontent.com/pykwon/python/refs/heads/master/testdata_utf8/rnn_test_toji.txt'
)

# 3. 텍스트 파일 읽기
with open(path_to_file, encoding='utf-8', errors='ignore') as obj:
    raw_text = obj.read()

print(raw_text[:100])
print('문자 수 : ', len(raw_text))

# 4. 텍스트 정제 함수 정의
def clean_str(text: str) -> str:
    text = re.sub(r"[^가-힣0-9() \n]", " ", text)
    text = re.sub(r"\s{2,}", " ", text)  # 공백이 2개 이상 반복되면 하나로 줄인다.
    return text.strip()                  # 앞뒤 공백을 제거해서 반환

# 5. 말뭉치 만들기
cleaned = clean_str(raw_text)    # 원문 텍스트를 정제한다.
corpus = cleaned.replace("\n", " [NL] ") # 줄바꿈을 [NL] 토큰으로 바꾼다.

# 6. 주요 설정값 지정
MAX_TOKENS = 3000     # 출력층 크기를 줄여 학습 속도를 높인다.
SEQ_LEN = 15          # 문맥 길이를 줄여 LSTM 계산량을 줄인다.
BATCH = 32            # 배치 크기를 줄여 메모리 부담을 낮춘다.
BUFFER = 2000         # 셔플 버퍼를 줄인다.
EMBED_DIM = 64        # 임베딩 차원을 줄인다.
LSTM_UNITS = 128      # LSTM 유닛 수를 줄인다.
EPOCHS = 2            # 짧게 학습한다.

# 7. TextVectorization 레이어 생성
vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,   # 자주 등장하는 단어 위주로 어휘 수를 제한한다.
    standardize=None,        # 이미 직접 정제했으므로 표준화 처리는 하지 않는다.
    split="whitespace",      # 공백 기준으로 단어를 나눈다.
    output_mode="int",       # 단어를 정수 ID로 바꾼다.
    output_sequence_length=None  # 문장 길이를 강제로 고정하지 않는다.
)

# 8. 단어 사전 학습
vectorizer.adapt(tf.data.Dataset.from_tensor_slices([corpus]))  # corpus를 기준으로 단어 사전을 만든다.

# 9. 단어 사전 확인
vocab = vectorizer.get_vocabulary()   # TextVectorization이 만든 단어 사전을 가져온다.
PAD = 0    # 0번 토큰은 패딩 토큰이다.
UNK = 1    # 1번 토큰은 사전에 없는 단어 토큰이다.
vocab_size = len(vocab)   # 실제 생성된 어휘 수를 구한다.

print(f'어휘 수 : {vocab_size} (PAD={PAD}, UNK={UNK})')
print('샘플 어휘 : ', vocab[:20])

# 10. 전체 corpus를 토큰 ID 시퀀스로 변환
token_ids = vectorizer(tf.constant([corpus]))[0].numpy() # corpus 전체를 정수 ID 배열로 바꾼다.
token_ids = token_ids.astype(np.int32)  # Dataset 입력용으로 int32 타입으로 바꾼다.
print('토큰 수 : ', len(token_ids))
print(token_ids[:20])
print(vocab[token_ids[0]])
print(vocab[token_ids[1]])
print(vocab[token_ids[2]])

# 11. 데이터 수 확인
if len(token_ids) <= SEQ_LEN + 1:
    raise ValueError('토큰 수가 너무 적어 학습할 수 없다.')

# 12. Dataset 생성
ds = tf.data.Dataset.from_tensor_slices(token_ids)         # 토큰 ID 배열을 Dataset으로 만든다.
ds = ds.window(SEQ_LEN + 1, shift=1, drop_remainder=True)  # 26개씩 묶고 한 칸씩 이동하는 윈도우를 만든다.
ds = ds.flat_map(lambda w: w.batch(SEQ_LEN + 1))           # 각 윈도우를 하나의 텐서로 변환한다.

# 13. 입력과 정답 분리 함수 정의
def split_xyFunc(chunk):
    x = chunk[:-1]   # 앞의 25개 토큰을 입력으로 사용한다.
    y = chunk[1:]    # 뒤의 25개 토큰을 정답으로 사용한다.
    return x, y      # 입력과 정답을 반환한다.

# 14. 학습용 데이터 파이프라인 구성
ds = ds.map(split_xyFunc, num_parallel_calls=tf.data.AUTOTUNE)   # 각 chunk를 x, y로 나눈다.
ds = ds.shuffle(BUFFER)                   # 데이터를 섞어서 학습 순서를 랜덤하게 만든다.
ds = ds.batch(BATCH, drop_remainder=True) # 지정한 크기만큼 배치로 묶는다.
ds = ds.prefetch(tf.data.AUTOTUNE)        # 다음 배치를 미리 준비해서 학습 속도를 높인다.

# 15. 학습 스텝 수 계산
windows = len(token_ids) - SEQ_LEN         # 만들 수 있는 전체 윈도우 수를 계산한다.
steps_per_epoch = min(100, max(1, windows // BATCH)) # 한 epoch에서 사용할 step 수를 계산한다.
print('steps_per_epoch : ', steps_per_epoch) # epoch당 step 수를 출력한다.

# 16. 모델 생성
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(SEQ_LEN,)), # 입력은 길이 25의 토큰 ID 시퀀스다.
    tf.keras.layers.Embedding(
        input_dim=vocab_size,   # 입력 가능한 토큰 ID 개수를 지정한다.
        output_dim=EMBED_DIM,   # 각 단어를 128차원 벡터로 바꾼다.
        mask_zero=True          # 0번 PAD 토큰은 학습에서 무시한다.
    ),
    tf.keras.layers.LSTM(
        LSTM_UNITS,        # LSTM의 은닉 상태 크기를 지정한다.
        return_sequences=True    # 각 시점마다 출력값을 반환한다.
    ),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(vocab_size)  # 각 시점마다 다음 단어 후보 전체에 대한 점수를 출력
])

# 17. 모델 컴파일
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True  # 모델 출력이 softmax 확률이 아니라 logits임을 알려준다.
)

model.compile(
    optimizer='adam',
    loss=loss_fn,
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name='acc')] # 정수 라벨용 정확도 지표를 사용한다.
)

model.summary()

# 18. ID를 토큰으로 바꾸기 위한 배열 생성
idx2tok = np.array(vocab, dtype=object)

# 19. 토큰 ID 목록을 텍스트로 바꾸는 함수 정의
def ids_to_text(ids):
    toks = idx2tok[np.array(ids, dtype=np.int32)]        # 토큰 ID들을 실제 단어로 바꾼다.
    toks = [("\n" if t == "[NL]" else t) for t in toks]  # [NL] 토큰은 실제 줄바꿈으로 복원한다.
    text = " ".join(toks)                # 단어 사이에 공백을 넣어 하나의 문자열로 만든다.
    text = text.replace(" \n ", "\n")    # 줄바꿈 앞뒤의 불필요한 공백을 정리한다.
    text = text.replace(" \n", "\n")
    text = text.replace("\n ", "\n")
    return text   # 복원된 텍스트를 반환

# 20. logits에서 다음 토큰 하나를 샘플링하는 함수 정의
def sample_from_logits(logits, temperature=1.0, top_k=30, forbid_ids=(0, 1)):
    logits = logits.astype(np.float64)

    for tid in forbid_ids:           # 금지할 토큰 ID를 하나씩 확인한다.
        if 0 <= tid < logits.size:   # 토큰 ID가 유효한 범위인지 확인한다.
            logits[tid] = -np.inf    # PAD, UNK가 선택되지 않도록 막는다.

    if temperature <= 0:             # temperature가 0 이하인지 확인한다.
        temperature = 1e-8           # 0으로 나누는 문제를 피하기 위해 작은 값으로 바꾼다.

    logits = logits / temperature    # temperature로 확률 분포를 조절

    if top_k is not None and top_k > 0:  # top_k가 지정되어 있는지 확인
        k = min(int(top_k), logits.size) # k가 전체 어휘 수를 넘지 않도록 제한

        if k < logits.size:        # k가 전체 어휘 수보다 작을 때만 후보를 제한
            top_idxs = np.argpartition(-logits, k - 1)[:k]  # 점수가 높은 상위 k개 토큰 ID를 찾는다.
            filtered = np.full_like(logits, -np.inf)        # 전체를 -inf로 채운 배열을 만든다.
            filtered[top_idxs] = logits[top_idxs]           # 상위 k개 토큰의 점수만 남긴다.
            logits = filtered                               # 후보가 제한된 logits로 교체

    logits = logits - np.max(logits)  # softmax 계산 안정성을 위해 최댓값을 뺀다.
    probs = np.exp(logits)            # logits를 양수 값으로 바꾼다.
    probs_sum = probs.sum()           # 확률 합을 계산한다.

    if probs_sum == 0 or not np.isfinite(probs_sum):  # 확률 합이 0이거나 비정상인지 확인한다.
        probs = np.ones_like(probs) / probs.size      # 문제가 있으면 균등 확률로 대체한다.
    else:
        probs = probs / probs_sum     # 확률 합이 1이 되도록 정규화한다.

    return np.random.choice(len(probs), p=probs) # 확률 분포에 따라 다음 토큰 ID를 하나 선택한다.

# 21. 문장 생성 함수 정의
def generateFunc(seed_text, max_new_tokens=80, temperature=0.9, top_k=30):
    seed = clean_str(seed_text)             # 시작 문장을 정제한다.
    seed = seed.replace("\n", " [NL] ")     # 시작 문장의 줄바꿈을 [NL] 토큰으로 바꾼다.
    seed_ids = vectorizer(tf.constant([seed]))[0].numpy().tolist() # 시작 문장을 토큰 ID 목록으로 바꾼다.

    context = [PAD] * max(0, SEQ_LEN - len(seed_ids))   # 시작 문장이 짧으면 왼쪽을 PAD로 채운다.
    context = context + seed_ids[-SEQ_LEN:]    # 최근 SEQ_LEN개 토큰만 문맥으로 사용한다.

    out_ids = []          # 새로 생성된 토큰 ID를 저장할 리스트를 만든다.

    for _ in range(max_new_tokens):      # 지정한 개수만큼 새 토큰을 생성한다.
        x = np.array(context, dtype=np.int32)[None, :]   # 현재 문맥을 모델 입력 형태인 2차원 배열로 바꾼다.
        pred = model.predict(x, verbose=0)     # 현재 문맥 다음에 올 단어 점수를 예측한다.
        logits = pred[0, -1]                   # 마지막 시점의 예측 결과만 가져온다.
        next_id = sample_from_logits(
            logits,
            temperature=temperature,
            top_k=top_k,
            forbid_ids=(PAD, UNK)
        )   # 다음 토큰 ID를 샘플링한다.

        out_ids.append(next_id)           # 생성된 토큰 ID를 결과 리스트에 추가한다.
        context = context[1:] + [next_id] # 가장 오래된 토큰을 빼고 새 토큰을 문맥에 추가한다.

    text = ids_to_text(out_ids)   # 생성된 토큰 ID들을 텍스트로 바꾼다.
    text = re.sub(r"\s{2,}", " ", text).strip()  # 여러 공백을 하나로 정리한다.
    return text

# 22. 학습 중 샘플 문장을 출력하는 콜백 정의
class SamplerCallback(tf.keras.callbacks.Callback):
    def __init__(self, sample_every=5):
        super().__init__()
        self.sample_every = sample_every    # 몇 epoch마다 샘플을 출력할지 저장한다.

    def on_epoch_end(self, epoch, logs=None):
        current_epoch = epoch + 1          # 화면에 보여줄 epoch 번호를 1부터 시작하게 만든다.
        total_epochs = self.params.get('epochs', current_epoch) # 전체 epoch 수를 가져온다.

        if current_epoch % self.sample_every != 0 and current_epoch != total_epochs:
            return          # 출력할 시점이 아니면 아무 작업도 하지 않는다.

        # 샘플 생성을 위한 시작 문장을 지정한다.
        seed = "귀녀의 모습을 한번 쳐다보고 떠나려 했다."
        sample = generateFunc(seed, max_new_tokens=80, temperature=0.9, top_k=30)

        print(f"\n[샘플 생성: {current_epoch} epoch]")   # 현재 epoch 정보를 출력한다.
        print(seed + " " + sample[:500])  # 시작 문장과 생성 결과 일부를 출력한다.

# 23. 모델 학습
history = model.fit(
    ds,                                  # 학습용 Dataset을 전달한다.
    epochs=EPOCHS,                       # 전체 학습 epoch 수를 지정한다.
    steps_per_epoch=steps_per_epoch,     # 한 epoch에서 실행할 step 수를 지정한다.
    callbacks=[SamplerCallback(sample_every=5)],  # 학습 중 샘플 생성을 수행할 콜백 객체를 전달
    verbose=2                            # epoch 단위로 학습 로그를 출력한다.
)

# 24. 최종 학습 결과 확인
print('final loss : ', float(history.history['loss'][-1]))  # 마지막 epoch의 손실값을 출력
print('final acc : ', float(history.history['acc'][-1]))    # 마지막 epoch의 정확도를 출력

# 25. 최종 문장 생성 테스트
seed = "귀녀의 모습을 한번 쳐다보고 떠나려 했다."    # 최종 테스트용 시작 문장을 지정
out = generateFunc(seed, max_new_tokens=200, temperature=0.8, top_k=50)

print('최종 결과 : \n')
print(seed + " " + out)      # 시작 문장과 생성된 문장을 함께 출력한다.